In [1]:
# Deterministic hash ordering for set()/dict iteration across runs
import os
os.environ["PYTHONHASHSEED"] = "0"

import pandas as pd
import numpy as np

observations = pd.DataFrame(pd.read_pickle('data/observations.pkl'))
pos_stars = sorted(set(observations[observations['has_exoplanets'] == 1]['star_name']))
neg_stars = sorted(set(observations[observations['has_exoplanets'] == 0]['star_name']))

# Sinusoidal positional encoding for BJD
NUM_FREQS = 8
min_period = 1.0
max_period = 7300.0
periods = np.logspace(np.log10(min_period), np.log10(max_period), NUM_FREQS)
freqs = 2.0 * np.pi / periods

def bjd_positional_encoding(bjd, ref_bjd):
    dt = bjd - ref_bjd
    encoding = []
    for f in freqs:
        encoding.append(np.sin(f * dt))
        encoding.append(np.cos(f * dt))
    return encoding

pos_inputs = []
neg_inputs = []

for star in pos_stars:
    star_obs = observations[observations['star_name'] == star].sort_values('bjd')
    ref_bjd = star_obs['bjd'].iloc[0]

    rows = []
    for idx in range(len(star_obs)):
        row = list(star_obs.iloc[idx])
        bjd = row[1]
        rv_centered = row[8]
        rv_err = row[3]
        exposure_time = row[4]
        rhkp = row[5]
        halpha = row[6]

        pos_enc = bjd_positional_encoding(bjd, ref_bjd)

        features = [rv_centered, rv_err, exposure_time, rhkp, halpha] + pos_enc
        rows.append(features)

    pos_inputs.append(np.array(rows))

for star in neg_stars:
    star_obs = observations[observations['star_name'] == star].sort_values('bjd')
    ref_bjd = star_obs['bjd'].iloc[0]

    rows = []
    for idx in range(len(star_obs)):
        row = list(star_obs.iloc[idx])
        bjd = row[1]
        rv_centered = row[8]
        rv_err = row[3]
        exposure_time = row[4]
        rhkp = row[5]
        halpha = row[6]

        pos_enc = bjd_positional_encoding(bjd, ref_bjd)

        features = [rv_centered, rv_err, exposure_time, rhkp, halpha] + pos_enc
        rows.append(features)

    neg_inputs.append(np.array(rows))

print(f"Positive stars: {len(pos_inputs)}, total observations: {sum(len(x) for x in pos_inputs)}")
print(f"Negative stars: {len(neg_inputs)}, total observations: {sum(len(x) for x in neg_inputs)}")
print(f"Feature dimension per observation: {pos_inputs[0].shape[1]}")

Positive stars: 430, total observations: 81585
Negative stars: 1596, total observations: 138733
Feature dimension per observation: 21


In [2]:
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix, f1_score, fbeta_score

def seed_everything(seed):
    import random, torch, numpy as np
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

star_to_seq = {}

pos_by_name = {s: seq for s, seq in zip(pos_stars, pos_inputs)}
neg_by_name = {s: seq for s, seq in zip(neg_stars, neg_inputs)}

star_to_label = {}
for star in pos_stars:
    star_to_seq[star] = pos_by_name[star]
    star_to_label[star] = 1
for star in neg_stars:
    star_to_seq[star] = neg_by_name[star]
    star_to_label[star] = 0

all_stars = sorted(star_to_seq.keys())
y = np.array([star_to_label[s] for s in all_stars], dtype=int)
all_seqs = [star_to_seq[s] for s in all_stars]
n_stars = len(all_stars)
print(f"Stars: {n_stars}, pos={y.sum()}, neg={(1-y).sum()}")

MAX_SEQ_LEN = 100

def truncate_star(s, max_len=MAX_SEQ_LEN):
    if len(s) > max_len:
        return s[:max_len]
    return s

all_seqs = [truncate_star(s) for s in all_seqs]
lens = [len(s) for s in all_seqs]
print(f"Sequence lengths after truncation: min={min(lens)}, max={max(lens)}, median={int(np.median(lens))}")


Using device: cuda
Stars: 2026, pos=430, neg=1596
Sequence lengths after truncation: min=18, max=100, median=41


In [3]:
class StarDataset(Dataset):
    def __init__(self, seqs, labels):
        self.data = seqs
        self.labels = labels
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        star = torch.tensor(self.data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return star, label

def collate_stars(batch):
    stars, labels = zip(*batch)
    max_len = max(s.shape[0] for s in stars)
    padded = []
    mask = []
    for star in stars:
        seq_len = star.shape[0]
        pad_len = max_len - seq_len
        feat_dim = star.shape[1]
        if pad_len > 0:
            padding = torch.zeros(pad_len, feat_dim)
            padded_star = torch.cat([star, padding], dim=0)
        else:
            padded_star = star
        star_mask = torch.cat([torch.ones(seq_len), torch.zeros(pad_len)])
        padded.append(padded_star)
        mask.append(star_mask)
    padded = torch.stack(padded)
    mask = torch.stack(mask)
    labels = torch.stack(labels)
    return padded, mask, labels

In [4]:
import torch.nn as nn
import torch.nn.functional as F

class AttentionPool(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attention = nn.Linear(d_model, 1)
    def forward(self, x, mask):
        scores = self.attention(x).squeeze(-1)
        scores = scores.masked_fill(~mask.bool(), float('-inf'))
        weights = F.softmax(scores, dim=1)
        pooled = (x * weights.unsqueeze(-1)).sum(dim=1)
        return pooled

class ExoplanetTransformer(nn.Module):
    def __init__(self, feat_dim=21, d_model=48, nhead=4, num_layers=1, dim_feedforward=96, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = AttentionPool(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1),
        )
    def forward(self, x, mask):
        x = self.input_proj(x)
        src_key_padding_mask = ~mask.bool()
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        x = self.pool(x, mask)
        out = self.classifier(x).squeeze(-1)
        return out

import math
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR

N_EPOCHS = 100
BATCH = 32
LR = 1e-3
WD = 5e-3
WARMUP = 5

def train_one_fold(train_idx, test_idx, rep_seed):
    seed_everything(rep_seed)
    train_seqs = [all_seqs[i] for i in train_idx]
    train_labels_list = [float(y[i]) for i in train_idx]
    test_seqs = [all_seqs[i] for i in test_idx]

    train_cat = np.concatenate(train_seqs, axis=0)
    feat_mean = train_cat.mean(axis=0)
    feat_std = train_cat.std(axis=0)
    feat_std = np.clip(feat_std, 1e-8, None)
    train_seqs = [(s - feat_mean) / feat_std for s in train_seqs]
    test_seqs = [(s - feat_mean) / feat_std for s in test_seqs]

    ds_tr = StarDataset(train_seqs, train_labels_list)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, drop_last=False,
                       collate_fn=collate_stars, pin_memory=(device.type=='cuda'))

    model = ExoplanetTransformer().to(device)
    n_pos_train = sum(train_labels_list)
    n_neg_train = len(train_labels_list) - n_pos_train
    pos_w = torch.tensor([n_neg_train / max(n_pos_train, 1)], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = Adam(model.parameters(), lr=LR, weight_decay=WD)

    def lr_fn(ep):
        if ep < WARMUP:
            return (ep + 1) / WARMUP
        progress = (ep - WARMUP) / (N_EPOCHS - WARMUP)
        return 0.5 * (1 + math.cos(math.pi * progress))
    sched = LambdaLR(opt, lr_fn)

    model.train()
    for ep in range(N_EPOCHS):
        for xb, mask_b, yb in dl_tr:
            xb, mask_b, yb = xb.to(device), mask_b.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb, mask_b), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
        sched.step()

    model.eval()
    ds_te = StarDataset(test_seqs, [0.0]*len(test_seqs))
    dl_te = DataLoader(ds_te, batch_size=BATCH, shuffle=False,
                       collate_fn=collate_stars, pin_memory=(device.type=='cuda'))
    all_logits = []
    with torch.no_grad():
        for xb, mask_b, _ in dl_te:
            xb, mask_b = xb.to(device), mask_b.to(device)
            logits = model(xb, mask_b).cpu().numpy()
            all_logits.append(logits)
    logits = np.concatenate(all_logits)
    logits = np.nan_to_num(logits, nan=0.0, posinf=35.0, neginf=-35.0)
    probs = 1.0 / (1.0 + np.exp(-logits))
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0).astype(np.float32)
    return probs


In [5]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix, f1_score, fbeta_score
N_REPS = 5
N_FOLDS = 5

all_oof_probs = np.zeros((n_stars, N_REPS))
all_oof_preds = np.zeros((n_stars, N_REPS), dtype=int)
rep_metrics = {'rep': [], 'pr_auc': [], 'roc_auc': [],
               'f1': [], 'f05': [], 'precision': [], 'recall': []}

for rep in range(N_REPS):
    oof_preds = np.zeros(n_stars, dtype=int)
    oof_probs = np.zeros(n_stars)
    rep_seed = 42 + rep
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep_seed)
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(np.zeros(n_stars), y)):

        inner_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=rep_seed)
        inner_probs_fold = np.zeros(len(train_idx), dtype=np.float32)
        for i_train, i_val in inner_skf.split(np.zeros(len(train_idx)), y[train_idx]):
            inner_probs_fold[i_val] = train_one_fold(train_idx[i_train], train_idx[i_val], rep_seed)

        vp_in, vr_in, vt_in = precision_recall_curve(y[train_idx], inner_probs_fold)
        vf1_in = 2 * vp_in * vr_in / (vp_in + vr_in + 1e-8)
        fold_thr = float(vt_in[int(np.argmax(vf1_in))]) if len(vt_in) > 0 else 0.5

        probs = train_one_fold(train_idx, test_idx, rep_seed)
        oof_probs[test_idx] = probs
        oof_preds[test_idx] = (probs >= fold_thr).astype(int)
        if (fold_idx + 1) % 2 == 0:
            print(f"  rep {rep} fold {fold_idx+1}/{N_FOLDS} done")
    all_oof_probs[:, rep] = oof_probs
    all_oof_preds[:, rep] = oof_preds

    roc = roc_auc_score(y, oof_probs)
    pr  = average_precision_score(y, oof_probs)
    cm = confusion_matrix(y, oof_preds)
    tn, fp, fn, tp = cm.ravel()
    prc = tp / (tp+fp) if (tp+fp)>0 else 0.0
    rec = tp / (tp+fn) if (tp+fn)>0 else 0.0
    f1  = f1_score(y, oof_preds, zero_division=0)
    f05 = fbeta_score(y, oof_preds, beta=0.5, zero_division=0)
    rep_metrics['rep'].append(rep); rep_metrics['pr_auc'].append(pr)
    rep_metrics['roc_auc'].append(roc); rep_metrics['f1'].append(f1)
    rep_metrics['f05'].append(f05)
    rep_metrics['precision'].append(prc); rep_metrics['recall'].append(rec)
    print(f"  rep {rep} (seed {rep_seed}): pr_auc={pr:.4f} roc_auc={roc:.4f} f1={f1:.4f} f0.5={f05:.4f} p={prc:.3f} r={rec:.3f}")

rep_df = pd.DataFrame(rep_metrics)
print(f"\naggregate (n={N_REPS} reps):")
for m in ['pr_auc','roc_auc','f1','f05','precision','recall']:
    v = rep_df[m].values
    print(f"  {m}: {v.mean():.4f} +/- {v.std(ddof=1):.4f} (min={v.min():.4f}, max={v.max():.4f})")

avg_oof = all_oof_probs.mean(axis=1)
combined_pr  = average_precision_score(y, avg_oof)
combined_roc = roc_auc_score(y, avg_oof)
combined_preds = (all_oof_preds.mean(axis=1) >= 0.5).astype(int)
combined_f1   = f1_score(y, combined_preds, zero_division=0)
combined_f05  = fbeta_score(y, combined_preds, beta=0.5, zero_division=0)
cm = confusion_matrix(y, combined_preds)
tn, fp, fn, tp = cm.ravel()

print("\ncombined oof (avg across reps):")
print(f"  pr_auc: {combined_pr:.4f}")
print(f"  roc_auc: {combined_roc:.4f}")
print(f"  f1: {combined_f1:.4f}  f0.5: {combined_f05:.4f}")
print(f"  p={tp/(tp+fp) if (tp+fp)>0 else 0:.3f}  r={tp/(tp+fn) if (tp+fn)>0 else 0:.3f} (TN={tn} FP={fp} FN={fn} TP={tp})")

def bootstrap_roc_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for ROC-AUC via the percentile method."""
    from sklearn.metrics import roc_auc_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap ROC-AUC")

    point = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aucs[i] = point  # fall back to point estimate if degenerate
            continue
        aucs[i] = roc_auc_score(yt, ys)
    lo = float(np.percentile(aucs, 2.5))
    hi = float(np.percentile(aucs, 97.5))
    return point, lo, hi

def bootstrap_pr_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for PR-AUC (average precision) via the percentile method."""
    from sklearn.metrics import average_precision_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap PR-AUC")

    point = float(average_precision_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aps = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aps[i] = point  # fall back to point estimate if degenerate
            continue
        aps[i] = average_precision_score(yt, ys)
    lo = float(np.percentile(aps, 2.5))
    hi = float(np.percentile(aps, 97.5))
    return point, lo, hi
pr_point, pr_lo, pr_hi = bootstrap_pr_auc(y, avg_oof)
roc_point, roc_lo, roc_hi = bootstrap_roc_auc(y, avg_oof)
print("\nbootstrap 95% ci (200 resamples):")
print(f"  pr_auc: {pr_point:.4f} [{pr_lo:.4f}, {pr_hi:.4f}]")
print(f"  roc_auc: {roc_point:.4f} [{roc_lo:.4f}, {roc_hi:.4f}]")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


  rep 0 fold 2/5 done
  rep 0 fold 4/5 done
  rep 0 (seed 42): pr_auc=0.2915 roc_auc=0.6238 f1=0.3907 f0.5=0.2932 p=0.251 r=0.877
  rep 1 fold 2/5 done
  rep 1 fold 4/5 done
  rep 1 (seed 43): pr_auc=0.2795 roc_auc=0.6252 f1=0.3908 f0.5=0.2951 p=0.254 r=0.851
  rep 2 fold 2/5 done
  rep 2 fold 4/5 done
  rep 2 (seed 44): pr_auc=0.3243 roc_auc=0.6824 f1=0.4381 f0.5=0.3462 p=0.304 r=0.786
  rep 3 fold 2/5 done
  rep 3 fold 4/5 done
  rep 3 (seed 45): pr_auc=0.3306 roc_auc=0.6963 f1=0.4167 f0.5=0.3237 p=0.282 r=0.800
  rep 4 fold 2/5 done
  rep 4 fold 4/5 done
  rep 4 (seed 46): pr_auc=0.3359 roc_auc=0.7008 f1=0.4381 f0.5=0.3448 p=0.302 r=0.798

aggregate (n=5 reps):
  pr_auc: 0.3124 +/- 0.0252 (min=0.2795, max=0.3359)
  roc_auc: 0.6657 +/- 0.0382 (min=0.6238, max=0.7008)
  f1: 0.4149 +/- 0.0237 (min=0.3907, max=0.4381)
  f05: 0.3206 +/- 0.0258 (min=0.2932, max=0.3462)
  precision: 0.2785 +/- 0.0252 (min=0.2513, max=0.3037)
  recall: 0.8223 +/- 0.0394 (min=0.7860, max=0.8767)

combined oo